# 01 Preliminary EDA

Notebook 01 is the first inspection of the NYC 311 training data. It checks the target definition, missing values, high-cardinality columns, date consistency, geographic fields, and possible leakage before any modeling choices are made.

The goal is not to train the final model here. The goal is to understand which columns are reliable enough to use later.


## How To Read This Notebook

Read this notebook as the evidence-gathering step. Each section answers a practical modeling question: which columns are complete enough to use, whether dates are valid, whether the target is balanced, and which fields could leak future information.

The final script does not copy every experiment here. It uses the conclusions from this inspection to build a cleaner production pipeline.


In [ ]:
# Imports used for the EDA tables and quick charts.
# pandas/numpy handle the tables; matplotlib handles the plots.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display, Markdown

# Keep summary tables fully visible during the first data pass.
# This avoids missing important columns because of notebook truncation.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


In [ ]:
# Load the training dataset.
# Let the notebook run from either the project root or the
# notebooks folder without manually changing file paths.
project_root = Path.cwd()

if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

# EDA uses the labeled training file because target and
# data-quality checks require columns such as Closed Date.
df = pd.read_csv(data_dir / "train.csv")

display(Markdown("### Dataset loaded"))
display(Markdown(f"The training dataset has been loaded from `{data_dir / 'train.csv'}`."))

# First sanity check: how many rows and raw columns
# are available before any cleaning or feature engineering.
display(pd.DataFrame({
    "metric": ["Rows", "Columns"],
    "value": [df.shape[0], df.shape[1]]
}))


In [ ]:
# Small display helpers keep the notebook layout consistent and readable.
# They do not transform data; they only standardize headings and table output.
def section(title, description=None):
    """Render a major notebook section with optional explanatory text."""
    display(Markdown(f"## {title}"))
    if description:
        display(Markdown(description))


def subsection(title, description=None):
    """Render a smaller subsection heading with optional explanatory text."""
    display(Markdown(f"### {title}"))
    if description:
        display(Markdown(description))


def show_table(data):
    """Display a pandas object using the notebook renderer."""
    display(data)


## Dataset Size And Shape

This first check confirms the number of rows, columns, and total cells. It gives context for later choices such as avoiding very wide one-hot encoding for high-cardinality text categories.


In [ ]:
# Dataset overview.
# Quick high-level profile before looking at individual columns.
section(
    "Dataset overview",
    "Quick overview of dataset size and memory usage."
)

n_rows, n_cols = df.shape

# Shape and total cell count confirm the CSV loaded correctly and give a sense
# of how large later operations will be.
total_cells = n_rows * n_cols

overview = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Total cells",
        "Memory usage MB"
    ],
    "value": [
        n_rows,
        n_cols,
        total_cells,
        round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

show_table(overview)


In [ ]:
# Column structure.
# Summarize data types, missingness, and cardinality for every field.
section(
    "Column structure",
    "Summary of data types, missing values, and unique values for each column."
)

# Count each pandas dtype to understand whether the raw table is mostly text,
# numeric, date-like strings, or a mix of formats.
dtype_summary = df.dtypes.value_counts().reset_index()
dtype_summary.columns = ["dtype", "count"]

subsection("Data type summary")
show_table(dtype_summary)

# Build one row per raw column with the indicators that matter for modeling:
# type, completeness, and number of unique values.
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notnull().sum().values,
    "missing_count": df.isnull().sum().values,
    "missing_%": (df.isnull().mean().values * 100).round(2),
    "unique_count": df.nunique(dropna=True).values,
    "unique_%": (df.nunique(dropna=True).values / len(df) * 100).round(2)
})

# Sort high-missing and high-cardinality columns first because they usually need
# the most careful preprocessing decisions.
column_summary = column_summary.sort_values(
    by=["missing_%", "unique_%"],
    ascending=[False, False]
)

subsection("Column summary")
show_table(column_summary)


## Missing Data Review

Missing values matter because a model can fail or learn misleading patterns if missingness is not handled consistently. The goal is to identify fields that may need dropping, imputation, or missingness indicator features.


In [ ]:
# Missing values.
# Identify columns and rows affected by missing data.
section(
    "Missing values",
    "Columns and rows affected by missing values. High missingness may require dropping, imputation, or special handling."
)

# Summarize missingness at the whole-dataset level before inspecting columns.
missing_summary = pd.DataFrame({
    "metric": [
        "Total missing cells",
        "Total missing %",
        "Rows with at least one missing value",
        "Rows with at least one missing value %"
    ],
    "value": [
        df.isnull().sum().sum(),
        round(df.isnull().sum().sum() / total_cells * 100, 2),
        df.isnull().any(axis=1).sum(),
        round(df.isnull().any(axis=1).mean() * 100, 2)
    ]
})

show_table(missing_summary)

# Sort missingness from highest to lowest so problematic columns appear first.
missing_df = column_summary[
    ["column", "missing_count", "missing_%"]
].sort_values("missing_%", ascending=False)

subsection("Columns sorted by missing values")
show_table(missing_df)

# Group columns into practical missingness bands to guide later cleaning choices.
missing_groups = pd.DataFrame({
    "group": [
        "More than 90% missing",
        "More than 50% missing",
        "Between 10% and 50% missing",
        "No missing values"
    ],
    "columns": [
        missing_df[missing_df["missing_%"] > 90]["column"].tolist(),
        missing_df[missing_df["missing_%"] > 50]["column"].tolist(),
        missing_df[(missing_df["missing_%"] >= 10) & (missing_df["missing_%"] <= 50)]["column"].tolist(),
        missing_df[missing_df["missing_count"] == 0]["column"].tolist()
    ]
})

subsection("Missingness groups")
show_table(missing_groups)


## Target Balance

This check compares requests closed within 24 hours with requests closed later. A very imbalanced target would make accuracy alone less reliable, so this check motivates using precision, recall, F1-score, and a confusion matrix later.


In [ ]:
# Target definition and distribution.
# The target is built here only for analysis; the final script recreates it the
# same way and does not feed Closed Date into the model.
section(
    "Target definition and distribution",
    "The target is 1 if the request was closed within 24 hours, and 0 otherwise. Missing, invalid, or negative closure times are assigned to class 0."
)

# Convert date strings to timestamps using the same format as the final script.
# Invalid date strings become NaT so they can be counted and handled consistently.
created = pd.to_datetime(
    df["Created Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)
closed = pd.to_datetime(
    df["Closed Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

# Resolution time is measured in hours because the business question uses a
# 24-hour threshold.
resolution_hours = (closed - created).dt.total_seconds() / 3600
valid_non_negative_resolution = resolution_hours.notna() & (resolution_hours >= 0)

# A row is positive only when the closure time exists, is not negative, and is
# within 24 hours of creation.
target_24h = (
    valid_non_negative_resolution
    & (resolution_hours <= 24)
).astype(int)

# Track target quality so invalid timestamps do not disappear silently.
target_quality = pd.DataFrame({
    "metric": [
        "Total rows",
        "Rows with missing Created Date",
        "Rows with missing Closed Date",
        "Rows with invalid resolution time",
        "Rows with negative resolution time",
        "Rows assigned target 1",
        "Rows assigned target 0"
    ],
    "value": [
        len(df),
        created.isna().sum(),
        closed.isna().sum(),
        resolution_hours.isna().sum(),
        (resolution_hours < 0).sum(),
        int(target_24h.sum()),
        int((target_24h == 0).sum())
    ]
})

subsection("Target quality")
show_table(target_quality)

# The target distribution gives the baseline class balance for later modeling.
target_distribution = target_24h.value_counts(dropna=False).sort_index().reset_index()
target_distribution.columns = ["target_24h", "count"]
target_distribution["percentage"] = (
    target_distribution["count"] / len(target_24h) * 100
).round(2)

subsection("Target distribution")
show_table(target_distribution)

target_distribution.set_index("target_24h")["count"].plot(
    kind="bar",
    figsize=(6, 4),
    title="Target distribution"
)

plt.xlabel("Closed within 24 hours")
plt.ylabel("Count")
plt.show()


In [ ]:
# Main categorical feature distributions.
# These columns are likely to explain large differences in service behavior.
section(
    "Main categorical feature distributions",
    "Most frequent values for key categorical variables, including dominant categories and imbalance inside features."
)

main_categorical_cols = [
    "Agency",
    "Problem (formerly Complaint Type)",
    "Borough"
]

for col in main_categorical_cols:
    if col in df.columns:
        subsection(col)

        # Show the most common values so dominant categories and rare-category
        # problems are visible before modeling.
        distribution = df[col].value_counts(dropna=False).head(15).reset_index()
        distribution.columns = [col, "count"]
        distribution["percentage"] = (
            distribution["count"] / len(df) * 100
        ).round(2)

        show_table(distribution)

        distribution.set_index(col)["count"].plot(
            kind="bar",
            figsize=(10, 4),
            title=f"Top values: {col}"
        )

        plt.xlabel(col)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


In [ ]:
# Temporal analysis.
# Check when complaints are created so later notebooks can engineer
# time-based features such as hour, weekday, and month.
section(
    "Temporal analysis",
    "Complaint creation timing. These patterns are useful for feature engineering."
)

# Reuse the parsed Created Date values from the target cell.
temporal_df = pd.DataFrame({
    "created_hour": created.dt.hour,
    "created_dayofweek": created.dt.dayofweek,
    "created_month": created.dt.month
})

day_mapping = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

temporal_df["created_day_name"] = temporal_df["created_dayofweek"].map(day_mapping)

subsection("Created hour distribution")

# Hourly counts reveal whether requests cluster during office hours or overnight.
hour_distribution = temporal_df["created_hour"].value_counts(dropna=False).sort_index().reset_index()
hour_distribution.columns = ["created_hour", "count"]
hour_distribution["percentage"] = (
    hour_distribution["count"] / len(temporal_df) * 100
).round(2)

show_table(hour_distribution)

hour_distribution.set_index("created_hour")["count"].plot(
    kind="bar",
    figsize=(10, 4),
    title="Requests by creation hour"
)

plt.xlabel("Hour")
plt.ylabel("Count")
plt.show()


subsection("Created day of week distribution")

# Reindexing keeps the weekday table in calendar order instead of frequency order.
day_distribution = temporal_df["created_day_name"].value_counts(dropna=False).reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
).reset_index()

day_distribution.columns = ["created_day", "count"]
day_distribution["percentage"] = (
    day_distribution["count"] / len(temporal_df) * 100
).round(2)

show_table(day_distribution)

day_distribution.set_index("created_day")["count"].plot(
    kind="bar",
    figsize=(9, 4),
    title="Requests by day of week"
)

plt.xlabel("Day of week")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


subsection("Created month distribution")

# Month counts can expose seasonality or partial-year coverage in the dataset.
month_distribution = temporal_df["created_month"].value_counts(dropna=False).sort_index().reset_index()
month_distribution.columns = ["created_month", "count"]
month_distribution["percentage"] = (
    month_distribution["count"] / len(temporal_df) * 100
).round(2)

show_table(month_distribution)

month_distribution.set_index("created_month")["count"].plot(
    kind="bar",
    figsize=(8, 4),
    title="Requests by creation month"
)

plt.xlabel("Month")
plt.ylabel("Count")
plt.show()


In [ ]:
# Date consistency check.
# Bad timestamps can distort the target, so they are counted explicitly here.
section(
    "Date consistency checks",
    "Checks whether Created Date and Closed Date are valid and consistent."
)

# Compare parsed dates against missing raw values to separate truly missing dates
# from values that are present but not parseable.
date_checks = pd.DataFrame({
    "metric": [
        "Invalid Created Date values",
        "Invalid Closed Date values",
        "Missing Closed Date values",
        "Rows where Closed Date is before Created Date",
        "Negative resolution times",
        "Resolution times greater than 1 year"
    ],
    "value": [
        created.isna().sum() - df["Created Date"].isna().sum(),
        closed.isna().sum() - df["Closed Date"].isna().sum(),
        closed.isna().sum(),
        (closed < created).sum(),
        (resolution_hours < 0).sum(),
        (resolution_hours > 24 * 365).sum()
    ]
})

show_table(date_checks)

# The distribution of closure time gives context for the 24-hour classification
# target and highlights extreme outliers.
resolution_summary = resolution_hours.describe().reset_index()
resolution_summary.columns = ["statistic", "resolution_time_hours"]

subsection("Resolution time summary")
show_table(resolution_summary)


In [ ]:
# Cardinality and feature difficulty.
# High-cardinality columns need careful treatment because they can act like IDs.
section(
    "Cardinality and feature difficulty",
    "Columns that may be difficult to use directly in modelling, such as ID-like or high-cardinality categorical variables."
)

# Cardinality is the number of distinct values in a column. Columns with nearly
# one unique value per row usually do not generalize well as direct features.
cardinality_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "unique_count": df.nunique(dropna=True).values,
    "unique_%": (df.nunique(dropna=True).values / len(df) * 100).round(2),
    "missing_%": (df.isnull().mean().values * 100).round(2)
}).sort_values("unique_count", ascending=False)

show_table(cardinality_df)

# Convert the cardinality table into modeling warnings that are easier to act on.
feature_difficulty = pd.DataFrame({
    "group": [
        "ID-like or almost unique columns",
        "Low-cardinality columns",
        "High-missing columns",
        "Potentially difficult categorical columns"
    ],
    "columns": [
        cardinality_df[cardinality_df["unique_%"] > 90]["column"].tolist(),
        cardinality_df[cardinality_df["unique_count"] <= 10]["column"].tolist(),
        cardinality_df[cardinality_df["missing_%"] > 50]["column"].tolist(),
        cardinality_df[
            (cardinality_df["dtype"] == "object") &
            (cardinality_df["unique_count"] > 50) &
            (cardinality_df["unique_%"] <= 90)
        ]["column"].tolist()
    ]
})

subsection("Feature difficulty groups")
show_table(feature_difficulty)


In [ ]:
# Geographic columns.
# Geographic fields are checked separately because they often contain missing or
# string-formatted numbers.
section(
    "Geographic columns",
    "Checks geographic variables and whether latitude/longitude can be converted to numeric values."
)

geo_cols = ["Latitude", "Longitude", "Location", "Incident Zip", "BBL"]
geo_info = []

for col in geo_cols:
    if col in df.columns:
        # Start each row with general quality indicators, then add numeric
        # conversion diagnostics for latitude/longitude when applicable.
        row = {
            "column": col,
            "dtype": df[col].dtype,
            "missing_%": round(df[col].isnull().mean() * 100, 2),
            "unique_values": df[col].nunique(dropna=True),
            "non_numeric_after_conversion": None,
            "min_after_conversion": None,
            "max_after_conversion": None
        }

        if col in ["Latitude", "Longitude"]:
            # Replace decimal commas before numeric conversion so coordinates like
            # "40,7" are interpreted as 40.7 instead of failing.
            converted = pd.to_numeric(
                df[col].astype(str).str.replace(",", ".", regex=False),
                errors="coerce"
            )

            row["non_numeric_after_conversion"] = converted.isnull().sum() - df[col].isnull().sum()
            row["min_after_conversion"] = converted.min()
            row["max_after_conversion"] = converted.max()

        geo_info.append(row)

geo_info_df = pd.DataFrame(geo_info)
show_table(geo_info_df)


## Leakage Review

Some columns may describe what happened after the complaint was created. Those columns can make validation look unrealistically good, so they should not be used as model inputs.


In [ ]:
# Potential data leakage.
# Leakage columns contain information that would not be known when making a new
# prediction and can make validation scores unrealistically high.
section(
    "Potential data leakage",
    "Columns that may contain post-creation information and should not be used as model features."
)

# Search column names for common post-event words. This is a screening step;
# each flagged column should still be checked with domain knowledge.
leakage_keywords = ["closed", "resolution", "status", "due"]

potential_leakage_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

leakage_df = pd.DataFrame({
    "potential_leakage_column": potential_leakage_cols,
    "reason": [
        "May contain information unavailable at prediction time"
        for _ in potential_leakage_cols
    ]
})

show_table(leakage_df)


## EDA Conclusions

The summary cell collects the main diagnostic findings so the next notebooks can focus on preprocessing and modeling rather than rediscovering basic dataset properties.


In [ ]:
# Final summary.
# Keep the main EDA findings in one place so later notebooks can reference the
# cleaning and feature-engineering decisions without rereading every section.
section(
    "Final diagnostic summary",
    "Main findings from the preliminary EDA."
)

final_summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Object columns",
        "Numeric columns",
        "Columns with more than 50% missing",
        "ID-like or almost unique columns",
        "Rows with missing Closed Date",
        "Rows assigned target 1",
        "Rows assigned target 0",
        "Majority class accuracy baseline"
    ],
    "value": [
        n_rows,
        n_cols,
        len(df.select_dtypes(include="object").columns),
        len(df.select_dtypes(include=np.number).columns),
        len(missing_df[missing_df["missing_%"] > 50]),
        len(cardinality_df[cardinality_df["unique_%"] > 90]),
        closed.isna().sum(),
        int(target_24h.sum()),
        int((target_24h == 0).sum()),
        round(target_24h.value_counts(normalize=True).max(), 4)
    ]
})

show_table(final_summary)

# Plain-language list of the main modeling issues found during EDA.
issues = pd.DataFrame({
    "main_issue": [
        "Many columns are stored as object dtype.",
        "Created Date and Closed Date need parsing before modelling.",
        "Closed Date is required to compute the target, but should not be used as a model feature.",
        "Missing or negative closure timestamps should be treated as not closed within 24 hours.",
        "Some columns have very high missingness.",
        "Some columns are ID-like or too high-cardinality for direct modelling.",
        "Categorical variables such as Agency, Complaint Type and Borough are likely important.",
        "Temporal features such as hour, day of week and month are likely useful.",
        "Geographic variables may require cleaning or conversion."
    ]
})

show_table(issues)
